In [6]:
import pandas as pd
import requests
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Load our baseline cricket data
df = pd.read_csv("cricket_base_dataset.csv")

# 1. UNIQUE VENUE GEOCODING
print("Geocoding unique venues... this may take a minute.")
geocoder = Nominatim(user_agent="cricket_weather_project_ml")
geocode = RateLimiter(geocoder.geocode, min_delay_seconds=1.5)

# Get unique combinations of venue and city
unique_venues = df[['venue', 'city']].drop_duplicates()

geo_dict = {}
for idx, row in unique_venues.iterrows():
    search_query = f"{row['venue']}, {row['city']}"
    print(f"Looking up: {search_query}")
    
    location = geocode(search_query)
    if not location:
        # Fallback to just city if specific stadium isn't found
        location = geocode(row['city'])
        
    if location:
        geo_dict[row['venue']] = (location.latitude, location.longitude)
    else:
        # Extreme fallback (0,0) if everything fails
        geo_dict[row['venue']] = (None, None)

# Map coordinates back to main dataframe
df['latitude'] = df['venue'].map(lambda x: geo_dict[x][0] if x in geo_dict else None)
df['longitude'] = df['venue'].map(lambda x: geo_dict[x][1] if x in geo_dict else None)

# Drop any rows where geocoding completely failed
df = df.dropna(subset=['latitude', 'longitude'])

# 2. FETCH WEATHER DATA FROM OPEN-METEO
print("\nFetching weather data from Open-Meteo API...")

temp_list = []
humidity_list = []
rain_48hr_list = []

# To avoid hitting the API too fast, we'll cache days we already looked up
weather_cache = {}

for idx, row in df.iterrows():
    date_str = row['date']
    lat = round(row['latitude'], 4)
    lon = round(row['longitude'], 4)
    innings = row['innings']
    
    # Create a unique cache key for (lat, lon, date)
    cache_key = (lat, lon, date_str)
    
    if cache_key not in weather_cache:
        # Build Open-Meteo Archive API URL (requests hourly data for that day)
        url = (f"https://archive-api.open-meteo.com/v1/archive?"
               f"latitude={lat}&longitude={lon}&start_date={date_str}&end_date={date_str}"
               f"&hourly=temperature_2m,relative_humidity_2m,precipitation&timezone=auto")
        
        try:
            response = requests.get(url).json()
            if 'hourly' in response:
                weather_cache[cache_key] = response['hourly']
            else:
                weather_cache[cache_key] = None
        except Exception as e:
            weather_cache[cache_key] = None
        time.sleep(1.0) # Polite spacing between API calls
        
    hourly_data = weather_cache[cache_key]
    
    if hourly_data:
        # Match time strategy: 
        # Innings 1 uses 3 PM (Hour 15). Innings 2 uses 8 PM (Hour 20)
        target_hour = 15 if innings == 1 else 20
        
        temp = hourly_data['temperature_2m'][target_hour]
        humidity = hourly_data['relative_humidity_2m'][target_hour]
        # Proxy for 48hr rain: Summing the full day's precipitation recorded here
        rain_48hr = sum(hourly_data['precipitation']) 
        
        temp_list.append(temp)
        humidity_list.append(humidity)
        rain_48hr_list.append(rain_48hr)
    else:
        temp_list.append(None)
        humidity_list.append(None)
        rain_48hr_list.append(None)

# Append new features to our dataframe
df['temperature_c'] = temp_list
df['humidity_pct'] = humidity_list
df['rain_48hr_mm'] = rain_48hr_list

# Drop rows with missing weather readings
df_master = df.dropna(subset=['temperature_c', 'humidity_pct'])

# Save the master dataset!
df_master.to_csv("cricket_weather_master.csv", index=False)
print("\n🎉 Phase 2 Complete! Master dataset saved as 'cricket_weather_master.csv'")
print(df_master[['match_id', 'innings', 'team', 'temperature_c', 'humidity_pct', 'team_total_score']].head(4))

Geocoding unique venues... this may take a minute.
Looking up: Kennington Oval, London, London
Looking up: Lord's, London, London
Looking up: Old Trafford, Manchester, Manchester
Looking up: Riverside Ground, Chester-le-Street, Chester-le-Street
Looking up: Headingley, Leeds, Leeds
Looking up: Boland Park, Paarl, Paarl
Looking up: Newlands, Cape Town, Cape Town
Looking up: Sabina Park, Kingston, Jamaica, Kingston
Looking up: SuperSport Park, Centurion, Centurion
Looking up: The Wanderers Stadium, Johannesburg, Johannesburg
Looking up: Narendra Modi Stadium, Ahmedabad, Ahmedabad
Looking up: VRA Ground, Amstelveen, Amstelveen
Looking up: Gaddafi Stadium, Lahore, Lahore
Looking up: Bay Oval, Mount Maunganui, Mount Maunganui
Looking up: Seddon Park, Hamilton, Hamilton
Looking up: Pallekele International Cricket Stadium, Kandy
Looking up: Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1), Al Amarat
Looking up: ICC Academy, Dubai, Dubai
Looking up: Sharjah Cricket Stadium, Sharjah
Look

In [1]:
import pandas as pd

# Load the master dataset from Phase 2
df = pd.read_csv("cricket_weather_master.csv")

# 1. CALCULATE OUR NEW TARGET VARIABLE (Run Rate)
# Avoid division by zero by ensuring overs_played > 0
df = df[df['overs_played'] > 0]
df['run_rate'] = df['team_total_score'] / df['overs_played']

# 2. ENCODE BATTING ORDER AS A BINARY FLAG
# 1 for Innings 1 (Batting First), 0 for Innings 2 (Batting Second)
df['is_batting_first'] = df['innings'].apply(lambda x: 1 if x == 1 else 0)

# 3. CREATE A SIMPLE TEAM STRENGTH PROXY
# Calculate the overall historical average run rate for each team in this dataset
team_strengths = df.groupby('team')['run_rate'].mean().to_dict()
df['team_avg_run_rate'] = df['team'].map(team_strengths)

# 4. CREATE AN INTERACTION FEATURE (Temperature x Batting First)
# This directly tests if afternoon heat affects batting first differently than night
df['temp_x_batting_first'] = df['temperature_c'] * df['is_batting_first']

# 5. SELECT FINAL FEATURES AND DROP MISSING VALUES
features = [
    'temperature_c', 
    'humidity_pct', 
    'rain_48hr_mm', 
    'is_batting_first', 
    'team_avg_run_rate',
    'temp_x_batting_first'
]
target = 'run_rate'

# Keep only what we need for modeling
df_ml = df[['match_id', 'date'] + features + [target]].dropna()

# Save this finalized clean dataset
df_ml.to_csv("cricket_ml_ready.csv", index=False)

print("features are ready.")
print(f"Final dataset shape: {df_ml.shape}")
print("\nSample rows of your ML training matrix:")
print(df_ml[['temperature_c', 'humidity_pct', 'is_batting_first', 'team_avg_run_rate', 'run_rate']].head(4))

features are ready.
Final dataset shape: (1148, 9)

Sample rows of your ML training matrix:
   temperature_c  humidity_pct  is_batting_first  team_avg_run_rate  run_rate
0           29.1            39                 1           6.032588  4.263566
1           27.5            39                 0           5.806152  5.846154
2           24.3            35                 1           6.032588  4.871287
3           21.2            37                 0           5.806152  3.743590
